In [1]:
###** Redouane Betrouni & Kernane Tewfik codes **##
## UNAIDS All Available Countries 1990-2023 HIV Prevalence Estimates Curve Fitting. 

# --- Imports and Setup ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit, dual_annealing
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold          # added: K-fold cross-validation for CV-MSE
from scipy.stats import qmc
import os
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
os.makedirs("plots", exist_ok=True)
os.makedirs("tables", exist_ok=True)

MAX_COUNTRIES = None ## set to an integer number when we want to test on the first few countries and None when to do All.

# --- Models ---
def logistic_model(t, K, r):
    y0 = prevalence[0] if 'prevalence' in globals() else 0.001
    return K / (1 + ((K - y0)/y0) * np.exp(-r * t))

def gompertz_model(t, K, r, t0):
    return K * np.exp(-np.exp(-r * (t - t0)))

def richards_model(t, K, r, t0, alpha, nu):
    return K / ((1 + alpha * np.exp(-r * (t - t0)))**(1/nu))

def ltvlms_model(t, K, r0, r1, nu):
    y0 = prevalence[0] if 'prevalence' in globals() else 0.001
    expo_term = np.clip(-nu * (r0 * t + 0.5 * r1 * t**2), -500, 500)
    inner = 1 + ((K**nu - y0**nu)/y0**nu) * np.exp(expo_term)
    return K / (inner**(1/nu))

# --- Metrics ---
# Standard goodness-of-fit +  Adjusted AIC  and Adjusted BIC that penalize Overfitting, more parameters more penalties.
# k = number of free params, so more complex models get penalised (that's the point).
def compute_metrics(y_true, y_pred, k):
    n = len(y_true)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    aic = n * np.log(mse) + 2 * k
    bic = n * np.log(mse) + k * np.log(n)
    adj_aic = aic + (2 * k * (k + 1)) / (n - k - 1)
    adj_bic = bic + (k * (k + 1)) / (n - k - 1)
    return rmse, r2, aic, bic, adj_aic, adj_bic


# --- Smart Bounds ---
def get_country_model_setup(prevalence, t):
    y0 = prevalence[0]
    max_prev = max(prevalence)
    t_min, t_max = t.min(), t.max()

    K_bounds = [y0 * 0.8, max_prev * 2]
    r_bounds = [0.0001, 1.0]
    t0_bounds = [t_min, t_max]
    alpha_bounds = [0.01, 2]
    nu_bounds = [0.05, 3]
    r_base = 1.0 / (t_max - t_min + 1)
    r1_center = -r_base / 2
    r1_range = abs(r1_center) * 5
    r1_bounds = [r1_center - r1_range, r1_center + r1_range]

    return {
        "logistic": [K_bounds, r_bounds],
        "gompertz": [K_bounds, r_bounds, t0_bounds],
        "richards": [K_bounds, r_bounds, t0_bounds, alpha_bounds, nu_bounds],
        "ltvlms": [K_bounds, r_bounds, r1_bounds, nu_bounds]
    }

# --- LHS ---
def latin_hypercube_init(bounds, n_samples=50):
    dim = len(bounds)
    sampler = qmc.LatinHypercube(d=dim)
    lhs_samples = sampler.random(n=n_samples)
    return np.array([
        bounds[i][0] + (bounds[i][1] - bounds[i][0]) * lhs_samples[:, i]
        for i in range(dim)
    ]).T

# --- Simulated Annealing for Richards ---
def fit_richards_annealing(t, y, bounds):
    def objective(params):
        try:
            pred = richards_model(t, *params)
            return mean_squared_error(y, pred)
        except:
            return 1e6  # fail-safe large error
    res = dual_annealing(objective, bounds=bounds, maxiter=300)
    return res.x if res.success else None

def multi_start_fit(model_func, t, y, bounds, n_starts=50):
    if model_func.__name__ == "richards_model":
        return fit_richards_annealing(t, y, bounds)
    best_score = np.inf
    best_params = None
    for guess in latin_hypercube_init(bounds, n_samples=n_starts):
        try:
            params, _ = curve_fit(model_func, t, y, p0=guess, bounds=np.array(bounds).T, maxfev=100000)
            pred = model_func(t, *params)
            score = mean_squared_error(y, pred)
            if score < best_score:
                best_score = score
                best_params = params
        except:
            continue
    return best_params

# --- Cross-Validation (K-fold) -----------------------------------------------
# Same scheme as the South Africa case study: 5 folds, shuffled with a fixed seed,
# multi-start fit on the training folds, and the validation MSE averaged over folds.
# It calls THIS notebook's multi_start_fit, so Richards is fit by annealing in the
# folds exactly as it is on the full data -> internally consistent per country.
def cross_validate(model_func, bounds, t, y, k_folds=5, n_starts=20):
    if len(t) < k_folds:                     # too few points to form k folds -> CV-MSE undefined
        return np.nan
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)
    mse_list = []
    for train_idx, test_idx in kf.split(t):
        t_train, t_test = t[train_idx], t[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        try:
            params = multi_start_fit(model_func, t_train, y_train, bounds, n_starts=n_starts)
            if params is None:
                continue
            pred = model_func(t_test, *params)
            mse_list.append(mean_squared_error(y_test, pred))
        except:
            continue
    return np.nanmean(mse_list) if mse_list else np.nan

# --- Plotting ---
def save_country_plot(code, name, years, prevalence, t, *params_list):
    plt.figure(figsize=(12,6))
    t_fine = np.linspace(t[0], t[-1], 200)
    years_fine = t_fine + years.min()
    plt.scatter(years, prevalence, label='Data', color='black', s=50, marker='x')
    models = [logistic_model, gompertz_model, richards_model, ltvlms_model]
    labels = ['Logistic', 'Gompertz', 'Richards', 'LTVLMS']
    styles = ['-', '-', '-', '--']
    for model, params, label, style in zip(models, params_list, labels, styles):
        if params is not None:
            plt.plot(years_fine, model(t_fine, *params), label=label, linestyle=style)
    plt.title(f"HIV Prevalence: {name} ({code})")
    plt.xlabel("Year")
    plt.ylabel("Prevalence (%)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"plots/{code}_comparison.png", dpi=300)
    plt.close()

# --- Main Execution ---
df = pd.read_csv("valid_prevalence_data.csv")
df = df.rename(columns={
    'Country_or_Region': 'CountryName',
    'Adults (15-49) prevalence (%)': 'Prevalence'
})
grouped = df.groupby(['Code', 'CountryName'])

for f in ["tables/all_countries_metrics.csv", "tables/all_countries_params.csv"]:
    if os.path.exists(f): os.remove(f)

for i, ((code, country_name), group) in enumerate(grouped):
    if MAX_COUNTRIES is not None and i >= MAX_COUNTRIES:
        print(f"\u26d4 Reached limit of {MAX_COUNTRIES} countries. Stopping early.")
        break
    try:
        print(f"\n\U0001f504 Processing {country_name} ({code})")
        group = group.sort_values('Year')
        years = group['Year'].values
        prevalence = group['Prevalence'].values
        t = years - years.min()

        bounds_dict = get_country_model_setup(prevalence, t)
        models = {
            "Logistic": logistic_model,
            "Gompertz": gompertz_model,
            "Richards": richards_model,
            "LTVLMS": ltvlms_model
        }

        params = {}
        preds = {}
        for name, model in models.items():
            print(f"\u2699\ufe0f Fitting {name}...")
            bounds = bounds_dict[name.lower()]
            fit = multi_start_fit(model, t, prevalence, bounds, n_starts=50)
            if fit is not None:
                params[name] = fit
                preds[name] = model(t, *fit)
            else:
                print(f"\u274c {name} model failed for {country_name}")

        if preds:
            metrics = []
            param_rows = []
            cv_scores = {}                                  # added: per-model CV-MSE for this country
            for name in preds:
                k = len(params[name])
                row = compute_metrics(prevalence, preds[name], k)
                metrics.append(row)

                # CV-MSE, computed the same way as the South Africa case study
                cv_scores[name] = cross_validate(models[name], bounds_dict[name.lower()], t, prevalence)

                param_dict = { 'Model': name, 'Code': code, 'CountryName': country_name }
                param_names = {
                    'Logistic': ['K', 'r'],
                    'Gompertz': ['K', 'r', 't0'],
                    'Richards': ['K', 'r', 't0', 'alpha', 'nu'],
                    'LTVLMS': ['K', 'r0', 'r1', 'nu']
                }[name]
                for pname, pval in zip(param_names, params[name]):
                    param_dict[pname] = pval
                param_rows.append(param_dict)

            metrics_df = pd.DataFrame(metrics, index=preds.keys(),
                columns=["RMSE", "R2", "AIC", "BIC", "Adj-AIC", "Adj-BIC"])
            metrics_df["CV-MSE"] = pd.Series(cv_scores)     # added: attach CV-MSE column
            params_df = pd.DataFrame(param_rows)

            save_country_plot(code, country_name, years, prevalence, t,
                *[params.get(name) for name in ['Logistic', 'Gompertz', 'Richards', 'LTVLMS']])

            metrics_df.reset_index(inplace=True)
            metrics_df.insert(0, 'Code', code)
            metrics_df.insert(1, 'CountryName', country_name)
            metrics_df.rename(columns={'index': 'Model'}, inplace=True)

            metrics_df.to_csv("tables/all_countries_metrics.csv", mode='a', header=not os.path.exists("tables/all_countries_metrics.csv"), index=False)
            params_df.to_csv("tables/all_countries_params.csv", mode='a', header=not os.path.exists("tables/all_countries_params.csv"), index=False)

    except Exception as e:
        print(f"\u274c Error in {country_name} ({code}): {e}")

print("\n\u2705 Processing complete.")



🔄 Processing Global (03M49WLD)
⚙️ Fitting Logistic...
⚙️ Fitting Gompertz...
⚙️ Fitting Richards...
⚙️ Fitting LTVLMS...

🔄 Processing Angola (AGO)
⚙️ Fitting Logistic...
⚙️ Fitting Gompertz...
⚙️ Fitting Richards...
⚙️ Fitting LTVLMS...

🔄 Processing Burundi (BDI)
⚙️ Fitting Logistic...
⚙️ Fitting Gompertz...
⚙️ Fitting Richards...
⚙️ Fitting LTVLMS...

🔄 Processing Benin (BEN)
⚙️ Fitting Logistic...
⚙️ Fitting Gompertz...
⚙️ Fitting Richards...
⚙️ Fitting LTVLMS...

🔄 Processing Burkina Faso (BFA)
⚙️ Fitting Logistic...
⚙️ Fitting Gompertz...
⚙️ Fitting Richards...
⚙️ Fitting LTVLMS...

🔄 Processing Bahamas (BHS)
⚙️ Fitting Logistic...
⚙️ Fitting Gompertz...
⚙️ Fitting Richards...
⚙️ Fitting LTVLMS...

🔄 Processing Belize (BLZ)
⚙️ Fitting Logistic...
⚙️ Fitting Gompertz...
⚙️ Fitting Richards...
⚙️ Fitting LTVLMS...

🔄 Processing Brazil (BRA)
⚙️ Fitting Logistic...
⚙️ Fitting Gompertz...
⚙️ Fitting Richards...
⚙️ Fitting LTVLMS...

🔄 Processing Botswana (BWA)
⚙️ Fitting Logistic...


In [2]:
# ============================================================
# Table 4-style averages + country-wise winner counts
# ============================================================

import numpy as np                      # added: NaN-safe winner selection for CV-MSE
import pandas as pd

metrics_all = pd.read_csv("tables/all_countries_metrics.csv")

# --- Table 4 style: mean metrics by model ---
# CV-MSE added so the averages table can report it alongside the other criteria.
table4_avg = (
    metrics_all
    .groupby("Model")[["RMSE", "R2", "AIC", "BIC", "Adj-AIC", "Adj-BIC", "CV-MSE"]]
    .mean()
    .reset_index()
)

table4_avg.to_csv("tables/table4_average_model_comparison.csv", index=False)

print("\n===== Table 4: Average Model Comparison Results =====")
print(table4_avg)


# --- Country-wise winner counts ---
# CV-MSE added to the "lower is better" criteria so the counts answer the reviewer's full list.
criteria_min = ["RMSE", "AIC", "BIC", "Adj-AIC", "Adj-BIC", "CV-MSE"]
criteria_max = ["R2"]

winner_rows = []

for (code, country), g in metrics_all.groupby(["Code", "CountryName"]):

    row = {
        "Code": code,
        "CountryName": country
    }

    for crit in criteria_min:
        col = g[crit]
        if col.notna().any():                       # skip if this criterion has no valid value (e.g. CV-MSE undefined)
            row[f"Best_{crit}"] = g.loc[col.idxmin(), "Model"]
        else:
            row[f"Best_{crit}"] = np.nan

    for crit in criteria_max:
        col = g[crit]
        if col.notna().any():
            row[f"Best_{crit}"] = g.loc[col.idxmax(), "Model"]
        else:
            row[f"Best_{crit}"] = np.nan

    winner_rows.append(row)

country_winners = pd.DataFrame(winner_rows)
country_winners.to_csv("tables/country_model_winners.csv", index=False)


# --- Count how many countries each model won ---
winner_count_tables = []

for col in [c for c in country_winners.columns if c.startswith("Best_")]:
    criterion = col.replace("Best_", "")

    counts = (
        country_winners[col]
        .value_counts()
        .rename_axis("Model")
        .reset_index(name=criterion)
    )

    winner_count_tables.append(counts)

model_win_counts = winner_count_tables[0]

for tbl in winner_count_tables[1:]:
    model_win_counts = model_win_counts.merge(tbl, on="Model", how="outer")

model_win_counts = model_win_counts.fillna(0)

# Force counts to be integers
for col in model_win_counts.columns:
    if col != "Model":
        model_win_counts[col] = model_win_counts[col].astype(int)

model_win_counts.to_csv("tables/model_win_counts.csv", index=False)

print("\n===== Country-wise Model Win Counts =====")
print(model_win_counts)



===== Table 4: Average Model Comparison Results =====
      Model      RMSE        R2         AIC         BIC     Adj-AIC  \
0  Gompertz  0.593352  0.396818  -86.817213  -82.238132  -86.017213   
1    LTVLMS  0.324190  0.788020 -156.093907 -149.988465 -154.714597   
2  Logistic  0.585287  0.441285 -122.493950 -119.441229 -122.106854   
3  Richards  0.574636  0.435803  -93.145910  -85.514107  -91.003052   

      Adj-BIC    CV-MSE  
0  -81.838132  1.102030  
1 -149.298810  0.373133  
2 -119.247681  1.082219  
3  -84.442678  1.042571  

===== Country-wise Model Win Counts =====
      Model  RMSE  AIC  BIC  Adj-AIC  Adj-BIC  CV-MSE  R2
0    LTVLMS    57   56   57       56       57      54  56
1  Richards     3    1    0        1        0       4   3
2  Gompertz     3    5    4        5        4       3   3
3  Logistic     0    1    2        1        2       2   1
